# LifeMate: timing notebook
Run all cells. Writes `/result/jupyter_submission.jsonl` (same format as `submission.jsonl`, with `time` in ms per sample) and `/result/jupyter_details.jsonl`. Same code and settings as `predict.sh`: everything comes from `/code/predict.py`.

In [ ]:
# Step 1: fixed seed
import os, random
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")  # vLLM in this process, as predict.sh
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)  # vLLM also gets seed=42 and greedy decoding (temperature 0) in predict.load_llm

In [ ]:
# Step 2: load the models (all inside the image, no internet, no API)
import sys, time
from pathlib import Path
sys.path.insert(0, "/code")
import predict as P

t = time.time()
ROOT, SPLIT = P.find_split(Path("/data"))  # /data may be the split folder itself or hold it
P.setup(ROOT, SPLIT, P.WEIGHTS)  # puts /data/<split> first on sys.path, so zaic_tools is the split's own copy
transcribe = P.load_asr(P.WEIGHTS / "asr_model")  # fine-tuned Gipformer 1.5, sherpa-onnx, CPU
generate = P.load_llm(P.WEIGHTS / "llm_model")  # fine-tuned Qwen3.5-4B, vLLM, float16, greedy
generate([{"role": "system", "content": "warm-up"}, {"role": "user", "content": "xin chào"}])
print(f"split {SPLIT}, load model: {time.time() - t:.1f}s")

In [ ]:
# Step 3: read the test cases
import json
from zaic_tools import ToolEnv

with open(ROOT / SPLIT / f"{SPLIT}.jsonl", encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]
print(len(test_cases), "test cases")

In [ ]:
# Step 4: predict one test case at a time. P.run starts its clock before ToolEnv and the audio, makes exactly one ToolEnv,
# runs ASR -> LLM -> env.call(...) -> LLM ..., and returns {"id", "tool_calls", "state_hash", "time"} (time in ms)
all_result, details = [], []
for item in test_cases:
    line, info = P.run(item, ROOT / SPLIT, SPLIT, ROOT, generate, transcribe)
    all_result.append(line)
    details.append(info)

Path("/result").mkdir(parents=True, exist_ok=True)
with open("/result/jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + chr(10))
with open("/result/jupyter_details.jsonl", "w", encoding="utf-8") as f:
    for d in details:
        f.write(json.dumps(d, ensure_ascii=False) + chr(10))
ms = sorted(r["time"] for r in all_result)
print(f"{len(ms)} lines -> /result/jupyter_submission.jsonl, mean {sum(ms) / len(ms):.0f} ms, max {ms[-1]} ms, {sum(m > 30000 for m in ms)} over 30 s")